# AG News Auto-Triage: Deep Learning for Business Operations
## Table of Contents
1. [Business Problem & Dataset Description](#1-business-problem--dataset-description)
2. [Exploratory Data Analysis](#2-exploratory-data-analysis)
3. [Preprocessing Pipeline](#3-preprocessing-pipeline)
4. [Model from Scratch (Keras)](#4-model-from-scratch-keras)
5. [Pretrained Model (DistilBERT)](#5-pretrained-model-distilbert)
6. [Hyperparameter Tuning](#6-hyperparameter-tuning)
7. [Model Evaluation & Interpretation](#7-model-evaluation--interpretation)
8. [Final Report](#8-final-report)

# 1. Business Problem & Dataset Description
## 1.1 Business Context
- A financial services firm receives 5,000 news articles daily.
- Analysts manually route each article to World, Sports, Business, or Sci/Tech desks.
- Time per article: 90 seconds. Analyst rate: $35/hour. Trading days: 250/year.
- Total labor cost: 5,000 × 90s × $35/hr × 250 days = $1.09M/year.
- A DL model automating 85% of routing saves ~$930K/year.

## 1.2 Dataset Reference
- Source: https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset/data
- 120,000 training samples, 7,600 test samples.
- 4 classes: World, Sports, Business, Sci/Tech.
- Features: article title + description (text), label (0-3).

In [ ]:
# Imports and environment setup
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Allow importing from src/ folder
sys.path.append(os.path.join(os.getcwd(), 'src'))

# Verify environment
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# 2. Exploratory Data Analysis

## 2.1 Load Data

In [ ]:
# We load the CSVs saved by src/setup_data.py
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

print("=== TRAIN SET ===")
print(train_df.head())
print(f"\nShape: {train_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")

print("\n=== TEST SET ===")
print(test_df.head())
print(f"\nShape: {test_df.shape}")


## 2.2 Class Distribution

In [ ]:
# Visualize how balanced the dataset is
plt.figure(figsize=(8, 5))
order = ['World', 'Sports', 'Business', 'Sci/Tech']
sns.countplot(data=train_df, x='category', order=order, palette='viridis')
plt.title('AG News Training Set: Class Distribution')
plt.xlabel('News Category')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('reports/class_distribution.png')  # Save for report
plt.show()

print("\nClass counts (train):")
print(train_df['category'].value_counts())


## 2.3 Text Length Analysis

In [ ]:
# Combine title + description for full article text
train_df['full_text'] = train_df['title'].astype(str) + ' ' + train_df['description'].astype(str)
test_df['full_text'] = test_df['title'].astype(str) + ' ' + test_df['description'].astype(str)

# Calculate length in words
train_df['word_count'] = train_df['full_text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 6))
for category in order:
    subset = train_df[train_df['category'] == category]['word_count']
    plt.hist(subset, bins=50, alpha=0.6, label=category)
plt.title('Distribution of Article Word Count by Category')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig('reports/word_count_distribution.png')
plt.show()

print(f"\nAverage word count per category:")
print(train_df.groupby('category')['word_count'].mean().round(1))


## 2.4 Word Clouds / Common Words

In [ ]:
# We use Python's Counter to find top words after basic cleaning
import re
from collections import Counter

def get_top_words(texts, n=10):
    """Join all texts, extract words, count frequencies."""
    all_text = ' '.join(texts.astype(str)).lower()
    # Keep only letters and spaces
    all_text = re.sub(r'[^a-z\s]', '', all_text)
    # Remove common stop words manually (beginner-friendly list)
    stop_words = {'the', 'and', 'for', 'are', 'but', 'not', 'you', 'all', 'can', 'had', 'her', 'was', 'one', 'our', 'out', 'day', 'get', 'has', 'him', 'his', 'how', 'man', 'new', 'now', 'old', 'see', 'two', 'way', 'who', 'boy', 'did', 'its', 'let', 'put', 'say', 'she', 'too', 'use', 'with', 'have', 'this', 'will', 'your', 'from', 'they', 'know', 'want', 'been', 'good', 'much', 'some', 'time', 'very', 'when', 'come', 'here', 'just', 'like', 'long', 'make', 'many', 'over', 'such', 'take', 'than', 'them', 'well', 'were', 'what', 'would', 'there', 'their', 'said', 'each', 'which', 'how', 'about', 'out', 'many', 'then', 'them', 'these', 'so', 'some', 'her', 'would', 'make', 'like', 'into', 'him', 'has', 'two', 'more', 'very', 'what', 'know', 'just', 'first', 'also', 'after', 'back', 'other', 'many', 'than', 'only', 'those', 'come', 'day', 'most', 'us'}
    words = [w for w in all_text.split() if w not in stop_words and len(w) > 2]
    return Counter(words).most_common(n)

plt.figure(figsize=(16, 10))
for i, category in enumerate(order, 1):
    texts = train_df[train_df['category'] == category]['full_text']
    top_words = get_top_words(texts, n=10)
    words, counts = zip(*top_words)
    
    plt.subplot(2, 2, i)
    sns.barplot(x=list(counts), y=list(words), palette='rocket')
    plt.title(f'Top Words: {category}')
    plt.xlabel('Frequency')
    
plt.tight_layout()
plt.savefig('reports/top_words_per_category.png')
plt.show()

print("\nEDA complete. Plots saved to reports/")


# 3. Preprocessing Pipeline
## 3.1 Text Cleaning
## 3.2 Tokenization & Vocabulary Building
## 3.3 Sequence Padding
## 3.4 Train/Validation Split

In [ ]:
# TODO: Import preprocessing functions from src.preprocessing
# TODO: Clean text, build tokenizer, convert to sequences
# TODO: Pad sequences to max_len
# TODO: Split training data into train/validation (90/10, stratified)

# 4. Model from Scratch (Keras)
## 4.1 Architecture: Embedding → BiLSTM → GlobalMaxPool → Dense → Softmax
## 4.2 Compilation & Training
## 4.3 History Tracking

In [ ]:
# TODO: Import build_scratch_model from src.model_builder
# TODO: Compile with Adam, sparse_categorical_crossentropy, accuracy
# TODO: Add EarlyStopping and ReduceLROnPlateau callbacks
# TODO: Train and save history
# TODO: Plot loss/accuracy curves using src.utils

# 5. Pretrained Model (DistilBERT Fine-Tuning)
## 5.1 Tokenization for BERT
## 5.2 Model Setup & Layer Freezing
## 5.3 Fine-Tuning & History

In [ ]:
# TODO: Import transformers (DistilBertTokenizer, TFDistilBertForSequenceClassification)
# TODO: Tokenize raw text for BERT (truncation, padding, max_length=128)
# TODO: Load pretrained model, freeze first 4 layers
# TODO: Compile with low learning rate (2e-5)
# TODO: Train with EarlyStopping, plot curves

# 6. Hyperparameter Tuning
## 6.1 Search Space
## 6.2 Best Configuration
## 6.3 Retrained Final Model

In [ ]:
# TODO: Import keras_tuner
# TODO: Define build_hp_model() with tunable embedding_dim, lstm_units, dropout, learning_rate
# TODO: Run RandomSearch or BayesianOptimization
# TODO: Print best hyperparameters
# TODO: Train final model with best config

# 7. Model Evaluation & Interpretation
## 7.1 Test Set Evaluation
## 7.2 Confusion Matrix
## 7.3 Per-Class Metrics
## 7.4 Scratch vs. BERT Comparison

In [ ]:
# TODO: Load both saved models from models/
# TODO: Evaluate on test set (model.evaluate)
# TODO: Generate predictions (model.predict)
# TODO: Classification report and confusion matrix for both models
# TODO: Create comparison dataframe (Accuracy, F1, Params, Training Time)

# 8. Final Report
## 8.1 Results Interpretation
## 8.2 Hardware & Memory Details
## 8.3 Next Steps
## 8.4 Lessons Learned
## 8.5 Team Contributions

### 8.1 Results Interpretation
*Write your analysis here after running the models. Discuss Business vs Sci/Tech confusion and business impact.*

### 8.2 Hardware & Memory Details
- CPU/GPU used:
- RAM available:
- Training time per epoch (scratch):
- Training time per epoch (BERT):
- Model size on disk:

### 8.3 Next Steps
*Suggestions for continuing the project: real-time API, RSS integration, multi-label support.*

### 8.4 Lessons Learned
*3-5 bullet points on technical and project management takeaways.*

### 8.5 Team Contributions
| Member | Contribution |
|--------|--------------|
| 1 | |
| 2 | |
| 3 | |
| 4 | |
| 5 | |
| 6 | |